# Pivot Tables & Merging DataFrames

Topics:
- `pivot_table()` — reshape data from long to wide
- `melt()` — reshape data from wide to long
- `merge()` — SQL-style joins (inner, left, right, outer)
- `concat()` — stacking DataFrames vertically or horizontally

In [ ]:
import pandas as pd
import numpy as np

## 1. Pivot Tables

```python
A pivot table **reshapes** data by rotating rows into columns (or columns into rows).
Same concept as Excel pivot tables — you're not changing the data, just changing how it's organized.

---

### Long vs Wide Format

Data can be stored in two shapes. The same information, different layout.

**Long format** — one row per observation
employee  month  sales
Alice     Jan    200
Alice     Feb    250
Alice     Mar    300
Bob       Jan    150
Bob       Feb    180


- Good for storage (less redundancy)
- Good for `groupby`, `merge`, plotting libraries
- Hard to read/compare across categories

**Wide format** — one row per entity, each category becomes a column
employee  Jan  Feb  Mar
Alice     200  250  300
Bob       150  180  160


- Good for human reading and comparison
- Easy to spot trends across columns
- Hard to do `groupby` on

---

### When to use `pivot_table()`

Use it when you want to answer questions like:
- "Show me sales per employee, broken down by month"
- "What's the average rating per department per quarter?"
- "Compare regions side by side across product categories"

Basically: whenever you want to turn a category column into column headers.

---

### Anatomy of `pivot_table()`

pd.pivot_table(
    df,
    index   = "employee",   # what becomes the rows
    columns = "month",      # what becomes the column headers
    values  = "sales",      # what fills the cells
    aggfunc = "sum"         # how to handle duplicates (sum, mean, count, max...)
)
aggfunc is required because there might be multiple rows matching the same
(index, column) combination — pandas needs to know how to collapse them into one cell.
```

In [ ]:
# Long format dataset — one row per (employee, month)
df = pd.DataFrame({
    "employee":   ["Alice", "Alice", "Alice", "Bob", "Bob", "Bob", "Carol", "Carol", "Carol"],
    "department": ["Eng",   "Eng",   "Eng",   "HR",  "HR",  "HR",  "Eng",   "Eng",   "Eng"],
    "month":      ["Jan",   "Feb",   "Mar",   "Jan", "Feb", "Mar", "Jan",   "Feb",   "Mar"],
    "sales":      [200,     250,     300,     150,   180,   160,   220,     270,     310],
    "rating":     [4.2,     4.5,     4.8,     3.9,   4.1,   3.8,   4.4,     4.6,     4.9]
})

df

,employee,department,month,sales,rating
0,Alice,Eng,Jan,200,4.2
1,Alice,Eng,Feb,250,4.5
2,Alice,Eng,Mar,300,4.8
3,Bob,HR,Jan,150,3.9
4,Bob,HR,Feb,180,4.1
5,Bob,HR,Mar,160,3.8
6,Carol,Eng,Jan,220,4.4
7,Carol,Eng,Feb,270,4.6
8,Carol,Eng,Mar,310,4.9


In [ ]:
# pivot: employee as rows, month as columns, sales as values
pd.pivot_table(
    df,
    index   = "employee",   # rows
    columns = "month",      # columns
    values  = "sales",      # cell values
    aggfunc = "sum"         # how to aggregate if duplicates
)



month,Feb,Jan,Mar
employee,,,
Alice,250,200,300
Bob,180,150,160
Carol,270,220,310


In [ ]:
# multiple index levels — rows = (department, employee)
pd.pivot_table(
    df,
    index   = ["department", "employee"],
    columns = "month",
    values  = "sales",
    aggfunc = "sum"
)

month                Feb  Jan  Mar
department employee               
Eng        Alice     250  200  300
           Carol     270  220  310
HR         Bob       180  150  160

In [ ]:
# multiple value columns
pd.pivot_table(
    df,
    index   = "employee",
    columns = "month",
    values  = ["sales", "rating"],
    aggfunc = "mean"
)

rating            sales              
month       Feb  Jan  Mar    Feb    Jan    Mar
employee                                      
Alice       4.5  4.2  4.8  250.0  200.0  300.0
Bob         4.1  3.9  3.8  180.0  150.0  160.0
Carol       4.6  4.4  4.9  270.0  220.0  310.0

In [ ]:
# add margins=True to get row/column totals
pd.pivot_table(
    df,
    index   = "employee",
    columns = "month",
    values  = "sales",
    aggfunc = "sum",
    margins = True          # adds "All" row and column
)

month,Feb,Jan,Mar,All
employee,,,,
Alice,250,200,300,750
Bob,180,150,160,490
Carol,270,220,310,800
All,700,570,770,2040


## 2. `melt()` — Wide to Long (Unpivot)

```python
`melt()` is the reverse of `pivot_table()` — it takes wide format and converts it back to long.
Think of it as "unpivoting" — column headers become values in a new row.

---

### Why you need it

In the real world, data often arrives in wide format (Excel reports, exported CSVs, survey results).
Most pandas operations (`groupby`, `merge`, plotting) expect long format.
`melt()` is how you fix that mismatch.

**Wide (what you received):**
employee  Jan  Feb  Mar
Alice     200  250  300
Bob       150  180  160



**Long (what pandas wants):**
employee  month  sales
Alice     Jan    200
Alice     Feb    250
Alice     Mar    300
Bob       Jan    150
...

---

### Anatomy of `melt()`

```python
df.melt(
    id_vars    = "employee",          # columns to keep as-is (the identifier)
    value_vars = ["Jan", "Feb", "Mar"], # columns to unpivot into rows
    var_name   = "month",             # name for the new 'column names' column
    value_name = "sales"              # name for the values column
)
id_vars — the anchor columns that repeat for every unpivoted row
value_vars — the columns being collapsed into a single column
if you omit value_vars, pandas melts every column not in id_vars

melt() and pivot_table() are inverses

# start with long → pivot → wide
wide = pd.pivot_table(df, index="employee", columns="month", values="sales", aggfunc="sum")

# wide → melt → long (back to original)
long = wide.reset_index().melt(id_vars="employee", var_name="month", value_name="sales")

Rule of thumb: after any operation that puts something in the index (pivot_table, groupby, set_index), call reset_index() before passing it to an operation that expects regular columns.

In [7]:
# wide format — one row per employee, one column per month
wide = pd.DataFrame({
    "employee": ["Alice", "Bob", "Carol"],
    "Jan":      [200,     150,   220],
    "Feb":      [250,     180,   270], 
    "Mar":      [300,     160,   310]
})

wide

,employee,Jan,Feb,Mar
0,Alice,200,250,300
1,Bob,150,180,160
2,Carol,220,270,310


In [14]:
# melt back to long format
long = wide.melt(
    id_vars    = "employee",   # columns to keep as-is
    value_vars = ["Jan", "Feb", "Mar"],  # columns to unpivot
    var_name   = "month",      # name for the new 'column names' column
    value_name = "sales"       # name for the values column
)
long


,employee,month,sales
0,Alice,Jan,200
1,Bob,Jan,150
2,Carol,Jan,220
3,Alice,Feb,250
4,Bob,Feb,180
5,Carol,Feb,270
6,Alice,Mar,300
7,Bob,Mar,160
8,Carol,Mar,310


## 3. Merging DataFrames — SQL-style Joins

`merge()` combines two DataFrames based on a shared key column.
Same as SQL JOIN.

| Join type | Keeps |
|-----------|-------|
| `inner`   | only rows that match in BOTH DataFrames |
| `left`    | all rows from left, matched rows from right (NaN if no match) |
| `right`   | all rows from right, matched rows from left (NaN if no match) |
| `outer`   | all rows from both, NaN where no match |


**Which to use:**
- `inner` — you only want clean, fully matched rows
- `left` — keep all your main data, enrich with extra info where available
- `right` — rare, same as left but flipped
- `outer` — auditing mismatches, finding what's missing on either side

In [15]:
employees = pd.DataFrame({
    "emp_id":     [1,       2,       3,       4],
    "name":       ["Alice", "Bob",   "Carol", "Dave"],
    "dept_id":    [10,      20,      10,      30]
})

departments = pd.DataFrame({
    "dept_id":   [10,    20,   40],
    "dept_name": ["Eng", "HR", "Marketing"]
})

print(employees)
print()
print(departments)

   emp_id   name  dept_id
0       1  Alice       10
1       2    Bob       20
2       3  Carol       10
3       4   Dave       30

   dept_id  dept_name
0       10        Eng
1       20         HR
2       40  Marketing


In [16]:
# INNER JOIN — only matching dept_ids
pd.merge(employees, departments, on="dept_id", how="inner")

,emp_id,name,dept_id,dept_name
0,1,Alice,10,Eng
1,2,Bob,20,HR
2,3,Carol,10,Eng


In [17]:
# LEFT JOIN 
pd.merge(employees, departments, on="dept_id", how="left")


,emp_id,name,dept_id,dept_name
0,1,Alice,10,Eng
1,2,Bob,20,HR
2,3,Carol,10,Eng
3,4,Dave,30,NaN


In [18]:
# RIGHT JOIN
pd.merge(employees, departments, on="dept_id", how="right")


,emp_id,name,dept_id,dept_name
0,1.0,Alice,10,Eng
1,3.0,Carol,10,Eng
2,2.0,Bob,20,HR
3,NaN,NaN,40,Marketing


```python
employees          departments
dept_id: 10,20,30  dept_id: 10,20,40
Right join says "keep ALL rows from the RIGHT table (departments)".

dept 10 → matched → Alice, Carol
dept 20 → matched → Bob
dept 40 → no match in employees → kept anyway, NaN for emp_id and name
Dave (dept 30) is dropped because dept 30 doesn't exist in the right table.

In [19]:
# OUTER JOIN — everything from both, NaN where no match
pd.merge(employees, departments, on="dept_id", how="outer")

,emp_id,name,dept_id,dept_name
0,1.0,Alice,10,Eng
1,3.0,Carol,10,Eng
2,2.0,Bob,20,HR
3,4.0,Dave,30,NaN
4,NaN,NaN,40,Marketing


In [20]:
# merging on columns with different names
salaries = pd.DataFrame({
    "employee_id": [1,      2,      3],
    "salary":      [95000,  70000,  88000]
})

pd.merge(employees, salaries, left_on="emp_id", right_on="employee_id")

,emp_id,name,dept_id,employee_id,salary
0,1,Alice,10,1,95000
1,2,Bob,20,2,70000
2,3,Carol,10,3,88000


```python
When the same key has different column names in each DataFrame, you can't use on= (that requires the same name in both). Instead you use left_on and right_on to tell pandas which column from each side to match on.


employees           salaries
emp_id  name        employee_id  salary
1       Alice       1            95000
2       Bob         2            70000
3       Carol       3            88000
The key is the same data (employee ID) but named differently — emp_id vs employee_id.


pd.merge(employees, salaries, left_on="emp_id", right_on="employee_id")
# "match employees.emp_id with salaries.employee_id"

#    emp_id   name  dept_id  employee_id  salary
# 0       1  Alice       10            1   95000
# 1       2    Bob       20            2   70000
# 2       3  Carol       10            3   88000
Notice both columns are kept (emp_id and employee_id) even though they contain the same values. You can drop the duplicate:


pd.merge(employees, salaries, left_on="emp_id", right_on="employee_id").drop(columns="employee_id")

## 4. Detecting Merge Issues

Always check your row count after a merge — unexpected duplicates or drops are common bugs.

In [21]:
# validate parameter — catches unexpected duplicates
# '1:1'  — each key appears once in both DataFrames
# '1:m'  — key unique in left, can repeat in right
# 'm:1'  — key can repeat in left, unique in right

pd.merge(employees, departments, on="dept_id", how="inner", validate="m:1")

,emp_id,name,dept_id,dept_name
0,1,Alice,10,Eng
1,2,Bob,20,HR
2,3,Carol,10,Eng


In [22]:
# indicator=True — adds a column showing where each row came from
pd.merge(employees, departments, on="dept_id", how="outer", indicator=True)

,emp_id,name,dept_id,dept_name,_merge
0,1.0,Alice,10,Eng,both
1,3.0,Carol,10,Eng,both
2,2.0,Bob,20,HR,both
3,4.0,Dave,30,NaN,left_only
4,NaN,NaN,40,Marketing,right_only


## 5. `concat()` — Stacking DataFrames

`concat()` stacks DataFrames — vertically (more rows) or horizontally (more columns).
No key matching needed — it just appends.

```python
### Key parameters

| Parameter | What it does | Default |
|-----------|-------------|---------|
| `axis` | `0` = stack vertically (more rows), `1` = stack horizontally (more columns) | `0` |
| `ignore_index` | resets index to 0,1,2... after stacking — avoids duplicate index values | `False` |
| `keys` | adds a label to identify which DataFrame each row came from (creates MultiIndex) | `None` |
| `join` | `"outer"` keeps all columns (NaN for missing), `"inner"` keeps only shared columns | `"outer"` |
| `verify_integrity` | raises error if the resulting index has duplicates | `False` |

---

In [ ]:
q1 = pd.DataFrame({"month": ["Jan", "Feb", "Mar"], "revenue": [1000, 1200, 1100]})
q2 = pd.DataFrame({"month": ["Apr", "May", "Jun"], "revenue": [1300, 1400, 1350]})

# vertical stack (axis=0, default) — more rows
pd.concat([q1, q2])

,month,revenue
0,Jan,1000
1,Feb,1200
2,Mar,1100
0,Apr,1300
1,May,1400
2,Jun,1350


In [24]:
# reset index after concat — otherwise you get duplicate index values
pd.concat([q1, q2], ignore_index=True)

,month,revenue
0,Jan,1000
1,Feb,1200
2,Mar,1100
3,Apr,1300
4,May,1400
5,Jun,1350


In [25]:
# add keys to track which DataFrame each row came from
pd.concat([q1, q2], keys=["Q1", "Q2"])

month  revenue
Q1 0   Jan     1000
   1   Feb     1200
   2   Mar     1100
Q2 0   Apr     1300
   1   May     1400
   2   Jun     1350

In [26]:
# horizontal stack (axis=1) — more columns
names   = pd.DataFrame({"name":   ["Alice", "Bob", "Carol"]})
scores  = pd.DataFrame({"score":  [95,      87,    91]})

pd.concat([names, scores], axis=1)

,name,score
0,Alice,95
1,Bob,87
2,Carol,91


## 6. `merge()` vs `concat()` — When to Use Which

| Situation | Use |
|-----------|-----|
| Combining rows of same structure (e.g. monthly files) | `concat()` |
| Adding columns from another table via a shared key | `merge()` |
| SQL-style JOIN logic | `merge()` |
| Stacking results from multiple queries | `concat()` |

## Practice Exercises

1. Create a pivot table showing average `rating` per `department` per `month` 
2. Melt the `wide` DataFrame but only unpivot Jan and Feb (leave Mar out)
3. Merge `employees` and `salaries` with a left join — which employee has no salary record?
4. Concat `q1` and `q2` and find the month with highest revenue using `idxmax()`

In [31]:
df = pd.DataFrame({
    "employee":   ["Alice", "Alice", "Alice", "Bob", "Bob", "Bob", "Carol", "Carol", "Carol"],
    "department": ["Eng",   "Eng",   "Eng",   "HR",  "HR",  "HR",  "Eng",   "Eng",   "Eng"],
    "month":      ["Jan",   "Feb",   "Mar",   "Jan", "Feb", "Mar", "Jan",   "Feb",   "Mar"],
    "sales":      [200,     250,     300,     150,   180,   160,   220,     270,     310],
    "rating":     [4.2,     4.5,     4.8,     3.9,   4.1,   3.8,   4.4,     4.6,     4.9]
})
df

,employee,department,month,sales,rating
0,Alice,Eng,Jan,200,4.2
1,Alice,Eng,Feb,250,4.5
2,Alice,Eng,Mar,300,4.8
3,Bob,HR,Jan,150,3.9
4,Bob,HR,Feb,180,4.1
5,Bob,HR,Mar,160,3.8
6,Carol,Eng,Jan,220,4.4
7,Carol,Eng,Feb,270,4.6
8,Carol,Eng,Mar,310,4.9


In [ ]:
# Exercise 1 — pivot table (avg rating per department per month)

wide = df.pivot_table(
    index = "department",
    columns = "month",
    values = "rating",
    aggfunc= "mean"
)
wide

month,Feb,Jan,Mar
department,,,
Eng,4.55,4.3,4.85
HR,4.10,3.9,3.80


In [ ]:
# Exercise 2 — melt only Jan and Feb
long = wide.melt(
    id_vars = "employee",
    value_vars = ["Jan", "Feb"],
    var_name = "month",
    value_name = "rating"
)
long

,employee,month,rating
0,Alice,Jan,200
1,Bob,Jan,150
2,Carol,Jan,220
3,Alice,Feb,250
4,Bob,Feb,180
5,Carol,Feb,270


In [38]:
# Exercise 3 — left merge, find employee with no salary record
employees = pd.DataFrame({
    "emp_id":  [1,       2,     3,       4],
    "name":    ["Alice", "Bob", "Carol", "Dave"],
    "dept_id": [10,      20,    10,      30]
})

salaries = pd.DataFrame({
    "employee_id": [1,     2,     3],       # Dave (emp_id=4) is missing
    "salary":      [95000, 70000, 88000]
})

merged = pd.merge(employees, salaries, left_on="emp_id", right_on="employee_id", how="left")
merged

,emp_id,name,dept_id,employee_id,salary
0,1,Alice,10,1.0,95000.0
1,2,Bob,20,2.0,70000.0
2,3,Carol,10,3.0,88000.0
3,4,Dave,30,NaN,NaN


In [41]:
# Exercise 4 — concat and find month with highest revenue
q1 = pd.DataFrame({"month": ["Jan", "Feb", "Mar"], "revenue": [1000, 1200, 1100]})
q2 = pd.DataFrame({"month": ["Apr", "May", "Jun"], "revenue": [1300, 1400, 1350]})

concatenated = pd.concat([q1, q2], ignore_index=True)
print(concatenated)
idx = concatenated["revenue"].idxmax()
highest_revenue = concatenated.loc[idx, "month"]
print(highest_revenue)

  month  revenue
0   Jan     1000
1   Feb     1200
2   Mar     1100
3   Apr     1300
4   May     1400
5   Jun     1350
May
